In [ ]:
from pathlib import Path
import json, math, re, time, hashlib, warnings, zipfile

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import shapely
from rasterio.windows import Window


# ============================================================
# 1. Configuration
# ============================================================

BASE = Path("../data")
POINTS_PANEL = Path("../data/study_grid_cells.csv")
GRID_DIR = BASE / "grids"
BOUNDARY_SHP = BASE / "smod_0514.shp"
POP_ROOT = Path("../data/raw/worldpop")

HEATWAVE_DIR = Path("../outputs/heatwave")
HEATWAVE_SCHEMES = ["local_p90", "local_p95", "city_p90", "city_p95", "fixed_tx35_tn25"]
HEAT_TYPES = ["day", "night", "compound"]

OUT_BASE = Path("../outputs/population_exposure")

POP_YEARS = list(range(2016, 2026))
EXPECTED_POINTS = 652
RETRIES = 4

FLAG_STUDY_FRACTION = 0.50


CONFIG = {
    "Abidjan_IvoryCoast": (32630, "Abidjan"),
    "AddisAbaba_Ethiopia": (32637, "Addis Ababa"),
    "Alexandria_Egypt": (32636, "Alexandria"),
    "Bamako_Mali": (32629, "Bamako"),
    "Bangalore_India": (32643, "Bangalore"),
    "Bangkok_Thailand": (32647, "Bangkok"),
    "BeloHorizonte_Brazil": (32723, "Belo Horizonte"),
    "Bogota_Colombia": (32618, "Bogota"),
    "BuenosAires_Argentina": (32721, "Buenos Aires"),
    "Cairo_Egypt": (32636, "Cairo"),
    "Chennai_India": (32644, "Chennai"),
    "Colombo_SriLanka": (32644, "Colombo"),
    "DaresSalaam_Tanzania": (32737, "Dar es Salaam"),
    "Delhi_India": (32643, "Delhi"),
    "Dhaka_Bangladesh": (32645, "Dhaka"),
    "Guayaquil_Ecuador": (32717, "Guayaquil"),
    "HoChiMinh_Vietnam": (32648, "Ho Chi Minh City"),
    "Hyderabad_India": (32644, "Hyderabad"),
    "Jakarta_Indonesia": (32748, "Jakarta"),
    "Johannesburg_SouthAfrica": (32735, "Johannesburg"),
    "Karachi_Pakistan": (32642, "Karachi"),
    "Kinshasa_DRC": (32734, "Kinshasa"),
    "KualaLumpur_Malaysia": (32647, "Kuala Lumpur"),
    "Lagos_Nigeria": (32631, "Lagos"),
    "Lahore_Pakistan": (32642, "Lahore"),
    "Lima_Peru": (32718, "Lima"),
    "Manaus_Brazil": (32720, "Manaus"),
    "Manila_Philippines": (32651, "Manila"),
    "MexicoCity_Mexico": (32614, "Mexico City"),
    "Mumbai_India": (32643, "Mumbai"),
    "Nairobi_Kenya": (32737, "Nairobi"),
    "PhnomPenh_Cambodia": (32648, "Phnom Penh"),
    "Santiago_Chile": (32719, "Santiago"),
    "SantoDomingo_DominicanRepublic": (32619, "Santo Domingo"),
    "SaoPaulo_Brazil": (32723, "Sao Paulo"),
}

CITY_TO_ISO3 = {
    "Abidjan_IvoryCoast": ["civ"], "AddisAbaba_Ethiopia": ["eth"], "Alexandria_Egypt": ["egy"],
    "Bamako_Mali": ["mli"], "Bangalore_India": ["ind"], "Bangkok_Thailand": ["tha"],
    "BeloHorizonte_Brazil": ["bra"], "Bogota_Colombia": ["col"], "BuenosAires_Argentina": ["arg"],
    "Cairo_Egypt": ["egy"], "Chennai_India": ["ind"], "Colombo_SriLanka": ["lka"],
    "DaresSalaam_Tanzania": ["tza"], "Delhi_India": ["ind"], "Dhaka_Bangladesh": ["bgd"],
    "Guayaquil_Ecuador": ["ecu"], "HoChiMinh_Vietnam": ["vnm"], "Hyderabad_India": ["ind"],
    "Jakarta_Indonesia": ["idn"], "Johannesburg_SouthAfrica": ["zaf"], "Karachi_Pakistan": ["pak"],
    "Kinshasa_DRC": ["cod", "cog"], "KualaLumpur_Malaysia": ["mys"], "Lagos_Nigeria": ["nga"],
    "Lahore_Pakistan": ["pak"], "Lima_Peru": ["per"], "Manaus_Brazil": ["bra"],
    "Manila_Philippines": ["phl"], "MexicoCity_Mexico": ["mex"], "Mumbai_India": ["ind"],
    "Nairobi_Kenya": ["ken"], "PhnomPenh_Cambodia": ["khm"], "Santiago_Chile": ["chl"],
    "SantoDomingo_DominicanRepublic": ["dom"], "SaoPaulo_Brazil": ["bra"],
}

POP_SEX = "t"
POP_AGES = ["00", "01", "05", "10", "15", "20", "25", "30", "35", "40",
            "45", "50", "55", "60", "65", "70", "75", "80", "85", "90"]
POP_PAT = re.compile(r"^(?P<iso3>[a-z]{3})_(?P<sex>[tmf])_(?P<age>\d{2})_(?P<year>\d{4})_.*\.tif$", re.I)

KEYS = ["city", "cell_id", "year"]


# ============================================================
# 2. Common functions
# ============================================================

def require(ok, message):
    if not ok:
        raise RuntimeError(message)


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda: f.read(1024 * 1024), b""):
            h.update(b)
    return h.hexdigest()


def save_csv(df, path):
    df.to_csv(path, index=False, encoding="utf-8-sig", float_format="%.17g")
    back = pd.read_csv(path, float_precision="round_trip")
    pd.testing.assert_frame_equal(df.reset_index(drop=True), back, check_dtype=False, check_exact=True)


def read_window(path, window):
    last = None
    for attempt in range(1, RETRIES + 1):
        try:
            with rasterio.open(path) as src:
                return src.read(1, window=window, boundless=False)
        except rasterio.errors.RasterioIOError as exc:
            last = exc
            time.sleep(2 * attempt)
    raise RuntimeError(f"Read still failing after {RETRIES} retries: {path} {window}") from last


def deg_area(geom):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        return float(geom.area)


def load_points():
    p = pd.read_csv(POINTS_PANEL)
    pts = p[["city", "cell_id", "lat", "lon"]].drop_duplicates().sort_values(["city", "cell_id"])
    require(len(pts) == EXPECTED_POINTS and not pts.duplicated(["city", "cell_id"]).any(),
            "Abnormal number or keys of study grid cells")
    require(set(pts.city) == set(CONFIG), "Grid-cell city list does not match the configuration")
    return pts


def load_boundaries():
    b = gpd.read_file(BOUNDARY_SHP)
    require(b.crs is not None and b.crs.to_epsg() == 4326, "Boundary CRS is not EPSG:4326")
    names = {v[1] for v in CONFIG.values()}
    require(b["name"].is_unique and set(b["name"]) == names, "Boundary name field does not map one-to-one to the 35 cities")
    out, fixed = {}, []
    for city, (_, name) in CONFIG.items():
        geom = b.loc[b["name"].eq(name), "geometry"].iloc[0]
        if not geom.is_valid:
            new = shapely.make_valid(geom)
            polys = [g for g in getattr(new, "geoms", [new]) if g.geom_type in ("Polygon", "MultiPolygon")]
            new = shapely.unary_union(polys)
            require(abs(deg_area(new) - deg_area(geom.buffer(0))) <= 1e-6 * deg_area(new),
                    f"{city} abnormal area change after boundary repair")
            fixed.append(dict(city=city, area_before=deg_area(geom.buffer(0)), area_after=deg_area(new)))
            geom = new
        require(geom.is_valid and not geom.is_empty and deg_area(geom) > 0, f"{city} invalid boundary")
        out[city] = geom
    return out, fixed


def load_city_grid(city, pts):
    g = gpd.read_file(GRID_DIR / f"{city}_era5_grid.gpkg")
    require(g.crs is not None and g.crs.to_epsg() == 4326 and g.cell_id.is_unique, f"{city} abnormal grid CRS/ID")
    p = pts[pts.city.eq(city)].sort_values("cell_id")
    g = g.set_index("cell_id").loc[p.cell_id].reset_index()
    require(np.allclose(g.lat_center, p.lat, atol=1e-6, rtol=0)
            and np.allclose(g.lon_center, p.lon, atol=1e-6, rtol=0), f"{city} grid-cell IDs do not match coordinates")
    g["cell_area_m2"] = g.to_crs(CONFIG[city][0]).geometry.area.to_numpy()
    g["cell_area_deg2"] = [deg_area(x) for x in g.geometry]
    return g


# ---------- WorldPop ----------

POP_INDEX_LOG = []


def build_pop_index():
    wanted = {iso for isos in CITY_TO_ISO3.values() for iso in isos}

    def parse(name):
        m = POP_PAT.match(name)
        if not m or m["iso3"].lower() not in wanted or m["sex"].lower() != POP_SEX:
            return None
        if int(m["year"]) not in POP_YEARS:
            return None
        return (m["iso3"].lower(), m["sex"].lower(), m["age"], int(m["year"]))

    index, sizes = {}, {}
    for p in POP_ROOT.rglob("*.tif"):
        key = parse(p.name)
        if key is None:
            continue
        if key not in index or len(str(p)) < len(str(index[key])):
            index[key], sizes[key] = p, p.stat().st_size

    for z in POP_ROOT.rglob("*_agesex_structures_*.zip"):
        if z.name[:3].lower() not in wanted:
            continue
        with zipfile.ZipFile(z) as zf:
            for info in zf.infolist():
                key = parse(Path(info.filename).name)
                if key is None:
                    continue
                vsi = f"/vsizip/{z.as_posix()}/{info.filename}"
                if key not in index:
                    index[key] = vsi
                    POP_INDEX_LOG.append(dict(file=Path(info.filename).name, source="zip_not_extracted"))
                elif not isinstance(index[key], str) and sizes[key] != info.file_size:
                    POP_INDEX_LOG.append(dict(file=Path(info.filename).name, source="zip_replaces_bad_extract",
                                              extracted=str(index[key]), extracted_bytes=sizes[key],
                                              zip_bytes=info.file_size))
                    index[key] = vsi
    return index


def available_isos(city, pop_index, missing_log):
    isos = []
    for iso3 in CITY_TO_ISO3[city]:
        need = [(iso3, POP_SEX, a, y) for a in POP_AGES for y in POP_YEARS]
        miss = [k for k in need if k not in pop_index]
        if not miss:
            isos.append(iso3)
        elif len(miss) == len(need) and iso3 != CITY_TO_ISO3[city][0]:
            missing_log.append(dict(city=city, iso3=iso3, missing_files=len(miss)))
            print(f"  [warning] {city} lacks all population files for {iso3}; population within that country is counted as 0 (see pop_nodata_share_study)", flush=True)
        else:
            raise RuntimeError(f"{city} lacks {len(miss)} population files, e.g. {miss[:3]}")
    return isos


def pop_grid(index, iso3):
    with rasterio.open(index[(iso3, POP_SEX, "00", POP_YEARS[0])]) as s:
        require(s.crs.to_epsg() == 4326, f"{iso3} population CRS is not EPSG:4326")
        t = s.transform
        require(t.b == 0 and t.d == 0 and t.e < 0, f"{iso3} population raster is rotated/flipped")
        return dict(x0=t.c, y0=t.f, rx=t.a, ry=-t.e)


def check_pop_alignment(path, g):
    with rasterio.open(path) as s:
        t = s.transform
        ok = (s.crs.to_epsg() == 4326 and np.isclose(t.a, g["rx"], rtol=0, atol=1e-12)
              and np.isclose(-t.e, g["ry"], rtol=0, atol=1e-12))
        kx = (t.c - g["x0"]) / g["rx"]
        ky = (g["y0"] - t.f) / g["ry"]
        ok = ok and abs(kx - round(kx)) < 1e-6 and abs(ky - round(ky)) < 1e-6
        require(ok, f"Population raster grid not aligned: {path}")
        return int(round(kx)), int(round(ky)), s.width, s.height, s.nodata


def pop_weights(geom, g):
    if geom.is_empty:
        return {}
    minx, miny, maxx, maxy = geom.bounds
    c0 = math.floor((minx - g["x0"]) / g["rx"]); c1 = math.floor((maxx - g["x0"]) / g["rx"])
    r0 = math.floor((g["y0"] - maxy) / g["ry"]); r1 = math.floor((g["y0"] - miny) / g["ry"])
    rr, cc = np.meshgrid(np.arange(r0, r1 + 1), np.arange(c0, c1 + 1), indexing="ij")
    rr, cc = rr.ravel(), cc.ravel()
    left = g["x0"] + cc * g["rx"]; top = g["y0"] - rr * g["ry"]
    cells = shapely.box(left, top - g["ry"], left + g["rx"], top)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        a = shapely.area(shapely.intersection(cells, geom)) / (g["rx"] * g["ry"])
    keep = a > 0
    return {(int(r), int(c)): float(w) for r, c, w in zip(rr[keep], cc[keep], a[keep])}


# ============================================================
# 3. Step 1: study area + population weights
# ============================================================

def study_area_for_cell(city, r, boundary, g_pop):
    cell = r.geometry
    study = cell.intersection(boundary)
    frac = deg_area(study) / r.cell_area_deg2 if not study.is_empty else 0.0
    row = dict(city=city, cell_id=int(r.cell_id), lat=float(r.lat_center), lon=float(r.lon_center),
               cell_area_m2=float(r.cell_area_m2), study_area_fraction=float(frac),
               study_area_m2=float(frac * r.cell_area_m2), study_area_km2=float(frac * r.cell_area_m2 / 1e6))
    return row, pop_weights(cell, g_pop), pop_weights(study, g_pop)


# ============================================================
# 4. Step 2: population (whole grid cell + within boundary)
# ============================================================

def city_population(city, isos, weights_full, weights_study, pop_index, g_pop):
    allkeys = set()
    for d in list(weights_full.values()) + list(weights_study.values()):
        allkeys |= set(d)
    require(allkeys, f"{city} has no population pixels")
    rmin = min(k[0] for k in allkeys); rmax = max(k[0] for k in allkeys)
    cmin = min(k[1] for k in allkeys); cmax = max(k[1] for k in allkeys)
    shape = (rmax - rmin + 1, cmax - cmin + 1)

    def read_pop(fp):
        kx, ky, w, h, nodata = check_pop_alignment(fp, g_pop)
        col_off, row_off = cmin - kx, rmin - ky
        require(col_off >= 0 and row_off >= 0 and col_off + shape[1] <= w and row_off + shape[0] <= h,
                f"City extent exceeds the population raster: {fp}")
        arr = read_window(fp, Window(col_off, row_off, shape[1], shape[0])).astype("float64")
        bad = ~np.isfinite(arr) | (arr < 0)
        if nodata is not None:
            bad |= arr == nodata
        return arr, bad

    def wsum(arr, weights):
        return sum(arr[r - rmin, c - cmin] * wv for (r, c), wv in weights.items())

    rows = []
    for year in POP_YEARS:
        total = np.zeros(shape, dtype="float64")
        covered = np.zeros(shape, dtype=bool)
        for iso3 in isos:
            for age in POP_AGES:
                arr, bad = read_pop(pop_index[(iso3, POP_SEX, age, year)])
                arr[bad] = 0.0
                total += arr
                if age == POP_AGES[0]:
                    covered |= ~bad
        nod = (~covered).astype("float64")
        for cell_id in weights_full:
            ws, wf = weights_study[cell_id], weights_full[cell_id]
            sw = sum(ws.values())
            rows.append(dict(city=city, cell_id=cell_id, year=year,
                             pop_cell_total=wsum(total, wf), pop_study=wsum(total, ws),
                             pop_nodata_share_study=(wsum(nod, ws) / sw) if sw > 0 else np.nan,
                             pop_nodata_share_cell=wsum(nod, wf) / sum(wf.values())))
    return rows


# ============================================================
# 5. Step 3: exposure
# ============================================================

def compute_exposure(pop):
    out = []
    for scheme in HEATWAVE_SCHEMES:
        f = HEATWAVE_DIR / f"annual_{scheme}_wide.csv"
        require(f.is_file(), f"Heatwave wide table missing: {f}")
        cols = [f"{m}_{t}" for t in HEAT_TYPES for m in ("HW_days", "HWDD")]
        hw = pd.read_csv(f, usecols=KEYS + ["annual_complete", "quality_status"] + cols)
        require(not hw.duplicated(KEYS).any(), f"{scheme} duplicate keys in heatwave table")
        x = pop.merge(hw, on=KEYS, how="left", validate="one_to_one")
        require(x.quality_status.notna().all(), f"{scheme} has grid-cell-years not found in the heatwave table")
        x.insert(3, "scheme", scheme)
        for t in HEAT_TYPES:
            x[f"exposure_people_days_{t}"] = x.pop_study * x[f"HW_days_{t}"]
            x[f"exposure_people_Cdays_{t}"] = x.pop_study * x[f"HWDD_{t}"]
        out.append(x)
    return pd.concat(out, ignore_index=True)


# ============================================================
# 6. Main program
# ============================================================

def run():
    pts = load_points()
    boundaries, fixed = load_boundaries()
    cities = list(CONFIG)
    pop_index = build_pop_index()
    print("Repaired invalid boundaries:", [f["city"] for f in fixed], flush=True)

    out = OUT_BASE
    out.mkdir(parents=True, exist_ok=True)
    summary = dict(status="running", output_directory=str(out), cities=cities,
                   points_panel=str(POINTS_PANEL), points_panel_sha256=sha256(POINTS_PANEL),
                   boundary=str(BOUNDARY_SHP), boundary_sha256=sha256(BOUNDARY_SHP), boundaries_repaired=fixed,
                   heatwave_dir=str(HEATWAVE_DIR) if HEATWAVE_DIR else None,
                   pop_years=POP_YEARS, retries=RETRIES)
    def save():
        (out / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    save()
    print("Output folder:", out, flush=True)

    SA, POP, missing_countries = [], [], []
    try:
        for i, city in enumerate(cities, 1):
            t0 = time.time()
            g = load_city_grid(city, pts)
            isos = available_isos(city, pop_index, missing_countries)
            g_pop = pop_grid(pop_index, isos[0])
            boundary = boundaries[city]

            # Step 1
            wf, ws = {}, {}
            for r in g.itertuples():
                row, w_full, w_study = study_area_for_cell(city, r, boundary, g_pop)
                row["pop_weight_full_sum"] = float(sum(w_full.values()))
                row["pop_weight_study_sum"] = float(sum(w_study.values()))
                wf[int(r.cell_id)], ws[int(r.cell_id)] = w_full, w_study
                SA.append(row)

            # Step 2
            POP.extend(city_population(city, isos, wf, ws, pop_index, g_pop))

            summary["cities_finished"] = i; save()
            print(f"[{i}/{len(cities)}] {city}: {len(g)} grid cells, took {time.time() - t0:.0f} s", flush=True)

        sa = pd.DataFrame(SA)
        sa["flag_low_study_fraction"] = sa.study_area_fraction < FLAG_STUDY_FRACTION

        pop = pd.DataFrame(POP).merge(sa[["city", "cell_id", "study_area_km2", "study_area_fraction"]],
                                      on=["city", "cell_id"], how="left", validate="many_to_one")
        pop["pop_study_density_km2"] = pop.pop_study / pop.study_area_km2.replace(0, np.nan)
        pop["pop_study_share"] = pop.pop_study / pop.pop_cell_total.replace(0, np.nan)

        require(len(sa) == EXPECTED_POINTS, "Abnormal number of rows in the study-area table")
        require(len(pop) == EXPECTED_POINTS * len(POP_YEARS), "Abnormal number of rows in the population table")
        require(not sa.duplicated(["city", "cell_id"]).any() and not pop.duplicated(KEYS).any(), "Duplicate output keys")

        if POP_INDEX_LOG:
            pd.DataFrame(POP_INDEX_LOG).to_csv(out / "population_files_read_from_zip.csv", index=False, encoding="utf-8-sig")
        save_csv(sa, out / "study_area_cells.csv")
        save_csv(pop, out / "population_study_area_2016_2025.csv")

        # Step 3
        expo = None
        if HEATWAVE_DIR is not None:
            expo = compute_exposure(pop)
            for scheme, d in expo.groupby("scheme", sort=False):
                save_csv(d.reset_index(drop=True), out / f"exposure_{scheme}_2016_2025.csv")

        # ---------- Summary ----------
        p20 = pop[pop.year.eq(2020)]
        summary.update(status="calculation_complete", study_area_rows=len(sa), population_rows=len(pop),
                       missing_countries=missing_countries,
                       pop_files_read_from_zip=len(POP_INDEX_LOG),
                       cells_outside_boundary=int((sa.study_area_fraction == 0).sum()),
                       cells_study_fraction_lt_flag=int(sa.flag_low_study_fraction.sum()),
                       cells_pop_nodata_share_study_gt_10pct=int((p20.pop_nodata_share_study > 0.10).sum()),
                       pop_study_total_2020=float(p20.pop_study.sum()),
                       pop_cell_total_2020=float(p20.pop_cell_total.sum()))
        if expo is not None:
            e = expo[expo.scheme.eq(HEATWAVE_SCHEMES[0])]
            summary["exposure_rows_per_scheme"] = int(len(e))
            summary["exposure_rows_with_heatwave_nan"] = int(e[f"HW_days_{HEAT_TYPES[0]}"].isna().sum())
        save()
    except (Exception, KeyboardInterrupt) as exc:
        summary.update(status="failed_or_interrupted", error=repr(exc)); save()
        raise

    print("Output folder:", out)
    return sa, pop, expo, out


SA_CELLS, POPULATION_SA, EXPOSURE_SA, SA_OUTPUT_DIR = run()